# Sally Prompt Analysis

Fetch recent extraction logs from the Sally admin API and analyze prompt/response quality.

**Requires:** `pip install requests pandas`

**Auth:** No auth needed for `localhost`. For `dev.spexxtool.com`, grab your session cookie from
the browser (DevTools → Application → Cookies → `session`) and paste it into `SESSION_COOKIE` below.

In [1]:
import requests
import pandas as pd
import json
from IPython.display import display, HTML

# Switch between local and dev:
BASE_URL = "http://localhost:8080"           # local
# BASE_URL = "https://dev.spexxtool.com"    # dev server

SESSION_COOKIE = ""   # only needed for dev server

session = requests.Session()
if SESSION_COOKIE:
    session.cookies.set("session", SESSION_COOKIE)

print(f"Targeting: {BASE_URL}")

ModuleNotFoundError: No module named 'pandas'

## Load recent logs

In [ ]:
resp = session.get(f"{BASE_URL}/admin/api/extraction-logs", params={"limit": 200})
resp.raise_for_status()
data = resp.json()

df = pd.DataFrame(data["logs"])
if df.empty:
    print("No logs found.")
else:
    # Convert types
    df["CreatedAt"] = pd.to_datetime(df["CreatedAt"])
    df["DurationMS"] = df["DurationMS"].astype(int)
    print(f"Loaded {len(df)} logs from {df['CreatedAt'].min().date()} to {df['CreatedAt'].max().date()}")
    display(df[["CreatedAt","PromptVersion","Provider","Model","Success","DurationMS",
                "PromptTokens","CompletionTokens","MissingFieldCount","PageURL"]].head(20))

## Summary stats

In [ ]:
total = len(df)
successes = df["Success"].sum()
print(f"Success rate: {successes}/{total} = {successes/total*100:.1f}%")
print(f"Avg duration (success): {df[df['Success']]['DurationMS'].mean():.0f} ms")
print(f"Avg prompt tokens:      {df['PromptTokens'].mean():.0f}")
print(f"Avg completion tokens:  {df['CompletionTokens'].mean():.0f}")
print(f"Avg missing fields:     {df['MissingFieldCount'].mean():.2f}")

In [ ]:
# Break down by prompt version — this is where A/B comparison lives
by_version = df.groupby("PromptVersion").agg(
    count=("Success", "count"),
    success_rate=("Success", "mean"),
    avg_dur_ms=("DurationMS", "mean"),
    avg_missing=("MissingFieldCount", "mean"),
    avg_prompt_tok=("PromptTokens", "mean"),
    avg_completion_tok=("CompletionTokens", "mean"),
).round(2)
by_version["success_rate"] = (by_version["success_rate"] * 100).round(1).astype(str) + "%"
display(by_version)

## Errors

In [ ]:
errors_df = df[~df["Success"]].copy()
print(f"{len(errors_df)} errors")
if not errors_df.empty:
    display(errors_df[["CreatedAt","PromptVersion","Provider","Error","PageURL"]].head(20))

## Inspect a single extraction (prompt + response)

Set `REQUEST_ID` to the `RequestID` of any row above.

In [ ]:
# Paste a RequestID from above
REQUEST_ID = df.iloc[0]["RequestID"] if not df.empty else ""

resp = session.get(f"{BASE_URL}/admin/api/extraction-logs/{REQUEST_ID}")
resp.raise_for_status()
log = resp.json()

print(f"Request:  {log['RequestID']}")
print(f"Version:  {log['PromptVersion']}")
print(f"Provider: {log['Provider']} / {log['Model']}")
print(f"Success:  {log['Success']}")
print(f"URL:      {log['PageURL']}")
if not log.get("Success"):
    print(f"Error:    {log['Error']}")

In [ ]:
# Show the prompt sent to the LLM
prompt_raw = log.get("PromptText", "")
try:
    prompt_parsed = json.loads(prompt_raw)
    print(json.dumps(prompt_parsed, indent=2))
except json.JSONDecodeError:
    print(prompt_raw)

In [ ]:
# Show the raw LLM response
resp_raw = log.get("ResponseText", "")
try:
    resp_parsed = json.loads(resp_raw)
    print(json.dumps(resp_parsed, indent=2))
except json.JSONDecodeError:
    print(resp_raw)

## A/B comparison

Once you have two prompt versions in the data, compare them side-by-side.
Set `VERSION_A` and `VERSION_B` to the `PromptVersion` strings you want to compare.

In [ ]:
VERSION_A = "extract-spec-v2"
VERSION_B = "extract-spec-v3"   # change when a new version exists

a_df = df[df["PromptVersion"] == VERSION_A]
b_df = df[df["PromptVersion"] == VERSION_B]

def stats(d, label):
    if d.empty:
        return {"version": label, "n": 0}
    return {
        "version": label,
        "n": len(d),
        "success_rate": f"{d['Success'].mean()*100:.1f}%",
        "avg_dur_ms": f"{d['DurationMS'].mean():.0f}",
        "avg_missing": f"{d['MissingFieldCount'].mean():.2f}",
        "avg_prompt_tok": f"{d['PromptTokens'].mean():.0f}",
        "avg_completion_tok": f"{d['CompletionTokens'].mean():.0f}",
    }

display(pd.DataFrame([stats(a_df, VERSION_A), stats(b_df, VERSION_B)]).set_index("version"))